# 01 · Explore and filter

The flat table is the engine for everything else. Filtering sets the
**basket** — the selection that later analyses and plots default to.

In [1]:
from pathlib import Path

from NanoOrganizer import open_project
from NanoOrganizer.demo import demo_root

PROJECT = demo_root("DemoProject")

# attach=False: the micrographs were linked in 00 and are in the saved store.
# Re-attaching by folder convention would work here and would be the wrong
# habit — see the note at the top of 00.
wb = open_project(PROJECT, attach=False)
print(wb.summary())


DemoProject: 6 samples, 12 measurements (12 readable here, 0 not mounted).
Modalities: uvvis, tem
Root: /home/yuzhang/Repos/OrgDemo/DemoProject


## Finding a column

The table gets wide, so search rather than scroll.

In [2]:
wb.columns("temperature") + wb.columns("status")

['synthesis.conditions.temperature_C',
 'synthesis.status',
 'synthesis.synthesis_batch.status']

## Filtering

`filter()` takes a pandas expression. Column names contain dots, so wrap them
in backticks. It returns the matching ids *and* makes them the basket.

In [3]:
wb.filter("`synthesis.conditions.temperature_C` >= 90")

['Sample000004', 'Sample000005', 'Sample000006']

One run failed. Excluding it is the usual first filter:

In [4]:
wb.filter("`synthesis.status` == 'done'")

['Sample000001',
 'Sample000002',
 'Sample000003',
 'Sample000004',
 'Sample000005']

In [5]:
cols = ["sample_id", "synthesis.conditions.temperature_C",
        "synthesis.batch_tag", "synthesis.status"]
wb.table()[cols]

,sample_id,synthesis.conditions.temperature_C,synthesis.batch_tag,synthesis.status
sample_id,,,,
Sample000001,Sample000001,60.0,demo_batch_1,done
Sample000002,Sample000002,70.0,demo_batch_1,done
Sample000003,Sample000003,80.0,demo_batch_1,done
Sample000004,Sample000004,90.0,demo_batch_2,done
Sample000005,Sample000005,100.0,demo_batch_2,done


## What can be run on the selection

The registry answers this, so neither a notebook nor the GUI keeps a list.

In [6]:
print("analyses:", wb.analyses())

for m in wb.measurements():
    print(f"  {m.measurement_id:40s} {m.group:10s} {len(m.paths) or m.pattern}")

analyses: ['curve_metrics', 'peak_fit', 'particle_sizing']
  Sample000001:uvvis:synthesis-spectra     curve      /home/yuzhang/Repos/OrgDemo/DemoProject/RawSpectra/demo_run/demo_b01_*.npy
  Sample000001:tem:characterization        image      3
  Sample000002:uvvis:synthesis-spectra     curve      /home/yuzhang/Repos/OrgDemo/DemoProject/RawSpectra/demo_run/demo_b02_*.npy
  Sample000002:tem:characterization        image      3
  Sample000003:uvvis:synthesis-spectra     curve      /home/yuzhang/Repos/OrgDemo/DemoProject/RawSpectra/demo_run/demo_b03_*.npy
  Sample000003:tem:characterization        image      3
  Sample000004:uvvis:synthesis-spectra     curve      /home/yuzhang/Repos/OrgDemo/DemoProject/RawSpectra/demo_run/demo_b04_*.npy
  Sample000004:tem:characterization        image      3
  Sample000005:uvvis:synthesis-spectra     curve      /home/yuzhang/Repos/OrgDemo/DemoProject/RawSpectra/demo_run/demo_b05_*.npy
  Sample000005:tem:characterization        image      3


## Clearing

An empty basket means *every* sample.

In [7]:
wb.clear()
len(wb.active)

6